# Working with the Claude API

## Making your first request to the Claude API.

Making your first request to the Anthropic API is straightforward once you understand the basic setup and structure. This notebook walks through the essential steps to get Claude responding to your prompts using Python.

### Setting Up Your Environment

Before making any API calls, you need to install the required packages & configure API Key.

First, install the necessary dependencies in your local Python environment:

```bash
uv add anthropic python_dotenv
```

Get your API Key from Claude Console → API Keys and add it to local `.env` file as

```
ANTHROPIC_API_KEY=your_anthropic_api_key
```

This approach keeps your API key out of your code and prevents accidentally committing it to version control. Always add .env to your .gitignore file.

Load the environment variables:

In [1]:
# load API key from local .env file

from dotenv import load_dotenv

load_dotenv(override=True)

True

### Create your API client:

In [2]:
# create our API client
import anthropic
from anthropic import Anthropic

print(f"Using Anthropic API version: {anthropic.__version__}")

client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"  # or "claude-sonnet-4-5"
MAX_TOKENS = 1024

Using Anthropic API version: 1.1.0


The core of making API requests is the `client.messages.create()` function. This function requires three key parameters:

<p align="center">
  <img src="../lessons/images/create_function.png"
  alt="The Create Function" width="450" height="250">
</p>

* `model` - The name of the Claude model you want to use
* `max_tokens` - A safety limit on response length (not a target)
* `messages` - The conversation history you're sending to Claude

The `max_tokens` parameter acts as a safety mechanism. If you set it to `1024`, Claude will stop generating after `1024` tokens _even if it has more to say_. Claude doesn't try to reach this limit - it just writes what it thinks is appropriate and stops if it hits the maximum.

### Understanding Messages

Messages represent the conversation between you and Claude, similar to a chat application. There are two types of messages:

<p align="center">
  <img src="images/understanding_messages.png" alt="Understanding Messages" width="450" height="250">
</p>

* `User messages` - Content you want to send to Claude (written by humans)
* `Assistant messages` - Responses that Claude has generated

Each message is a dictionary with a role (either "user" or "assistant") and content (the actual text).

### Making Your First Request

Here's a complete example of making a request to Claude:

In [4]:
# call the API
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[
        {
            "role": "user",
            "content": "What is Quantum computing? Answer in 1 sentence",
        }
    ],
)

When you run this code, Claude will process your request and return a response object containing the generated text along with metadata about the request.

In [6]:
# what did we get?
response

Message(id='msg_011CfGPBwJYzJndQcEpeuCQd', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computing that uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing quantum computers to solve certain complex problems much faster than classical computers.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=42, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

### Extracting the Response

The response object contains a lot of information, but you usually just want the generated text. Access it using:

```
message.content[0].text
```

This gives you clean, readable output like: _"Quantum computing is a type of computation that leverages quantum mechanics principles like superposition and entanglement to process information using quantum bits (qubits), potentially solving certain complex problems exponentially faster than classical computers."_

With these basics in place, you can start experimenting with different prompts and building more complex interactions with Claude.

In [7]:
# what we usually want is just the content of the message
response.content[0].text

'Quantum computing is a type of computing that uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing quantum computers to solve certain complex problems much faster than classical computers.'

In [8]:
# and this is how we get the tokens used
response.usage.input_tokens, response.usage.output_tokens

(19, 42)

## Multi-Turn Conversations

When working with the Anthropic API and Claude, there's a crucial concept you need to understand: **Claude doesn't store any of your conversation history**. Each request you make is completely independent, with no memory of previous exchanges.

This means if you want to have a multi-turn conversation where Claude remembers context from earlier messages, you need to handle the conversation state yourself.

### The Problem with Stateless Conversations

Let's say you ask Claude "What is quantum computing?" and get a good response. Then you follow up with "Write another sentence" - Claude has no idea what you're referring to. It will write a sentence about something completely random because it has no memory of the quantum computing discussion.

<p align="center">
  <img src="images/multi_turn_conversations.png" alt="Multi-turn Conversations" width="350" height="200">
</p>

### How Multi-Turn Conversations Work

To maintain conversation context, you need to do two things:

* Manually maintain a list of all messages in your code
* Send the complete message history with every request

<p align="center">
  <img src="images/multi_turn_conv1.png" alt="Multi-turn Conversations-1" width="400" height="250">
</p>

Here's the flow that actually works:

1. Send your initial user message to Claude
2. Take Claude's response and add it to your message list as an assistant message
3. Add your follow-up question as another user message
4. Send the entire conversation history to Claude

<p align="center">
  <img src="images/multi_turn_conv2.png" alt="Multi-turn Conversations-2" width="400" height="250">
</p>

### Building Helper Functions

To simulate a conversation, where Claude should know what we asked before so it can _continue_ from there, we need to pass back the entire message history on every `client.messages.create(...)` call.

Let's simulate that now - first we'll create some helper functions.

In [9]:
def add_user_message(history, message):
    history.append({"role": "user", "content": message})
    return history


def add_assistant_message(history, message):
    history.append({"role": "assistant", "content": message})
    return history


def chat(history):
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=history,
    )
    return response.content[0].text

    # return (
    #     response.content[0].text,
    #     response.usage.input_tokens,
    #     response.usage.output_tokens,
    # )

Here's how you use these functions to maintain a conversation:

In [10]:
# we are using the rich.console package for generating
# colorful text only - it's not really needed for API calls
from rich.console import Console

console = Console(force_jupyter=False)


message_history = []  # start with a blank
messages = [
    "Tell me about Quantum computing in 1 sentence",
    "Write another sentence about it",
]

for msg in messages:
    message_history = add_user_message(message_history, msg)
    console.print(f"[blue]User:[/blue] {msg}")
    response_text = chat(message_history)
    message_history = add_assistant_message(message_history, response_text)
    console.print(f"[green]AI:[/green] {response_text}[yellow]\n------\n[/yellow]")

User: Tell me about Quantum computing in 1 sentence
AI: Quantum computers harness the bizarre properties of quantum mechanics—where
particles can exist in multiple states simultaneously—to process certain types 
of problems exponentially faster than classical computers.
------

User: Write another sentence about it
AI: Unlike classical computers that process information as 1s and 0s, quantum 
computers use quantum bits (qubits) that can be 0, 1, or both at the same time,
enabling them to explore many possible solutions in parallel.
------



Now Claude will understand that "Write another sentence" refers to expanding on the quantum computing definition, because you've provided the complete conversation context.

These helper functions will be useful throughout your work with Claude, making it much easier to build applications that can maintain meaningful conversations over multiple exchanges.

Here's another _classic_ example

In [11]:
# here's another chat session. Again, rich.console used only for colorful text
from rich.console import Console

console = Console(force_jupyter=False)

message_history2 = []
messages = [
    "Hi, my name is Manish. What's your name?",
    "Tell me a joke about Generative AI",
    "What's my name again?",
]

for msg in messages:
    message_history2 = add_user_message(message_history2, msg)
    console.print(f"[blue]User:[/blue] {msg}")
    response_text = chat(message_history2)
    message_history2 = add_assistant_message(message_history2, response_text)
    console.print(f"[green]AI:[/green] {response_text}[yellow]\n------\n[/yellow]")

User: Hi, my name is Manish. What's your name?
AI: Hi Manish! I'm Claude, an AI assistant made by Anthropic. Nice to meet you!
How can I help you today?
------

User: Tell me a joke about Generative AI
AI: Here's one for you:

Why did the AI go to therapy?

Because it had too many layers to work through! 🧠

---

Or if you prefer something a bit different:

A generative AI walks into a bar. The bartender asks, "What'll you have?"

The AI says, "I don't know, what did everyone else order?"
------

User: What's my name again?
AI: Your name is Manish! You told me at the start of our conversation. 😊
------



### Exercise

Try this exercise with the Claude API

Make a chatbot with the three helper functions above
1. Prompt the user to enter some message using built-in `input` function
2. Add it to our list of messages
3. Call the API
4. Add generated text to list of messages
5. Print generated text
6. Repeat from 1 until user enters "bye"

In [12]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [13]:
# create our API client
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"  # or "claude-sonnet-4-5"
MAX_TOKENS = 1024

In [14]:
# our helper functions
def add_user_message(history, message):
    history.append({"role": "user", "content": message})
    return history


def add_assistant_message(history, message):
    history.append({"role": "assistant", "content": message})
    return history


def chat(history):
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=history,
    )
    return response.content[0].text

> 📌 **NOTE** Running the cell below will popup a text input box into which you must enter your input. You may not notice it immediately, especially if you are running this notebook inside of VS Code.
>
> This is an endless loop, until you type `bye` into the text input!

In [15]:
# we are using the rich text formatting library here, just to add some
# color to text displayed.
# Add it to your env using `uv add rich`

from rich.console import Console

console = Console(force_jupyter=False)

history = []

while True:
    user_message = input("User: ")
    user_message = user_message.strip()
    if user_message.lower() in ["exit", "quit", "bye"]:
        console.print("\n[#FF7B72]Exiting chat...[/#FF7B72]")
        break

    console.print(f"\n[blue]User:[/blue] {user_message}")
    history = add_user_message(history, user_message)
    assistant_message = chat(history)
    console.print(
        f"[green]Assistant:[/green] {assistant_message}[yellow]\n------\n[/yellow]"
    )
    history = add_assistant_message(history, assistant_message)


User: Hi, my name is Bob. What's your name?
Assistant: Hi Bob! I'm Claude, an AI assistant made by Anthropic. Nice to meet 
you! How can I help you today?
------


User: Which is the largest planet in the solar system?
Assistant: Jupiter is the largest planet in the solar system. It's a gas giant 
with a diameter of about 88,846 miles (142,984 kilometers), which makes it 
roughly 11 times wider than Earth. Jupiter is also by far the most massive 
planet in our solar system.
------


User: Can you give me 5 facts about it? Display it as a bulleted list
Assistant: Here are 5 facts about Jupiter:

• **Massive size** - Jupiter is so large that over 1,300 Earths could fit 
inside it

• **Great Red Spot** - Jupiter has a giant storm called the Great Red Spot that
has been raging for at least 300 years and is larger than Earth

• **Many moons** - Jupiter has 95 known moons, more than any other planet in 
our solar system

• **Fast rotation** - Despite its enormous size, Jupiter completes one

## System Prompts

System prompts are a powerful way to customize how Claude responds to user input. Instead of getting generic answers, you can shape Claude's tone, style, and approach to match your specific use case.

<p align="center">
  <img src="images/sys_prompt1.png" alt="System Prompts" width="350" height="200">
</p>

### Why System Prompts Matter

Consider building a math tutor chatbot. When a student asks `"How do I solve 5x + 2 = 3 for x?"`, you want Claude to act like a real tutor, not just spit out the answer. A good math tutor should:

* Initially give hints rather than complete solutions
* Patiently walk students through problems step by step
* Show solutions for similar problems as examples

You definitely don't want Claude to:

* Immediately give direct answers
* Tell students to just use a calculator

### How System Prompts Work

<p align="center">
  <img src="images/how_sys_prompts_work.png" alt="System Prompts" width="350" height="200">
</p>

System prompts provide Claude with guidance on how to respond. You define them as plain strings and pass them into the create function call. The key benefits are:

* System prompts provide Claude guidance on how to respond
* Claude will try to respond in the same way someone in the specified role would respond
* Helps keep Claude on task

Here's the basic structure:

In [16]:
from rich.console import Console

console = Console(force_jupyter=False)

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

messages = [
    {"role": "user", "content": "What is 2 + 2?"},
]

response = client.messages.create(
    model=MODEL,
    messages=messages,
    max_tokens=MAX_TOKENS,
    system=system_prompt,
)

console.print(f"[green]Assistant:[/green] {response.content[0].text}")

Assistant: Great question! Let me guide you through this step by step.

First, think about what it means to add. When we add, we're combining groups of
things together.

**Let me ask you:** If you have 2 apples in one hand, and 2 apples in your 
other hand, how many apples do you have total when you put them together?

Try visualizing it, or even counting on your fingers if that helps!


And let's try the same **WITHOUT** a system prompt

In [17]:
from rich.console import Console

console = Console(force_jupyter=False)

messages = [
    {"role": "user", "content": "What is 2 + 2?"},
]

response = client.messages.create(
    model=MODEL,
    messages=messages,
    max_tokens=MAX_TOKENS,
)

console.print(f"[green]Assistant:[/green] {response.content[0].text}")

Assistant: 2 + 2 = 4


### Seeing the Difference

Without a system prompt, Claude gives the final answer. This might be helpful, but it doesn't encourage the student to think through the problem themselves.

With the math tutor system prompt, Claude's response changes dramatically. Instead of providing the final answer, Claude provides guidance and encourages the student to arrive at the final response.

## Building a Flexible Chat Function

Rather than hard-coding system prompts, we can make our `chat` function more reusable by accepting system prompts as parameters, like so:

In [18]:
def chat(messages, system=None):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

This approach handles an important detail: Claude's API doesn't accept `system=None`, so you need to conditionally include the system parameter only when it's provided.

Now you can call your chat function with or without a system prompt:

In [19]:
response = chat(messages)
console.print(f"[green]Assistant:[/green] {response}")

Assistant: 2 + 2 = 4


In [20]:
# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
response = chat(messages, system=system)
console.print(f"[green]Assistant:[/green] {response}")

Assistant: Great question! Let me guide you through this step by step.

First, let's think about what addition means. When we add numbers, we're 
combining groups of things together.

**Here's my first question for you:** If you have 2 apples in one hand, and 2 
apples in your other hand, how many apples do you have in total?

Try visualizing this, or you could even use objects like blocks or coins to 
help you count them all together. What do you get?


System prompts are essential for creating AI applications that behave consistently and appropriately for their intended purpose. They transform generic AI responses into specialized, role-appropriate interactions.

## System Prompt Exercise

Let's go through a quick exercise on using System prompts. Consider the following example where we are asking Claude to generate a Python function that checks a string for duplicate characters. Like so:

In [21]:
messages = []

user_msg = """
Write a Python function that checks a string for duplicate characters. 
"""

console.print(f"[blue]User:[/blue] {user_msg}")
messages = add_user_message(messages, user_msg)
response = chat(messages)
console.print(f"[green]Assistant:[/green] {response}")

User: 
Write a Python function that checks a string for duplicate characters. 

Assistant: # Python Function to Check for Duplicate Characters

Here are several approaches, from simple to more efficient:

## 1. **Simple Approach (Using Set)**
```python
def has_duplicates(string):
    """
    Check if a string has duplicate characters.
    Returns True if duplicates exist, False otherwise.
    """
    return len(string) != len(set(string))

# Examples
print(has_duplicates("hello"))      # True (l appears twice)
print(has_duplicates("python"))     # False (all unique)
print(has_duplicates("aabbcc"))     # True
```

## 2. **Return Duplicate Characters**
```python
def find_duplicates(string):
    """
    Return a set of duplicate characters in the string.
    """
    seen = set()
    duplicates = set()
    
    for char in string:
        if char in seen:
            duplicates.add(char)
        else:
            seen.add(char)
    
    return duplicates

# Examples
print(find_duplicates("

Notice that this generates a ton of code & comments. We want to write a system prompt that generates as consice an output as possible.

In [22]:
system_prompt = """
You are an expert Python developer who writes clear & concise code.
Provide just 1 response to coding questions - just 1 code listing. 
No doc strings required. No explanation required,
"""

messages = []

user_msg = """
Write a Python function that checks a string for duplicate characters. 
"""

console.print(f"[blue]User:[/blue] {user_msg}")
messages = add_user_message(messages, user_msg)
response = chat(messages, system=system_prompt)
console.print(f"[green]Assistant:[/green] {response}")

User: 
Write a Python function that checks a string for duplicate characters. 

Assistant: 
```python
def has_duplicates(s: str) -> bool:
    return len(s) != len(set(s))
```


## Temperature

**Temperature** is a powerful parameter that **controls how predictable or creative Claude's responses will be**. Understanding how to use it effectively can dramatically improve your AI applications.

### How Claude Generates Text

Before diving into temperature, it helps to understand Claude's text generation process. When you send Claude a prompt like "What do you think?", it goes through three key steps:

* `Tokenization` - Breaking your input into smaller chunks
* `Prediction` - Calculating probabilities for possible next words
* `Sampling` - Choosing a token based on those probabilities

<p align="center">
  <img src="../lessons/images/how_claude_generates_response.png" alt="How Claude Generates Response" width="640" height="350">
</p>

In this example, Claude might assign a `30%` probability to "about", `20%` to "would", `10%` to "of", and so on. The model then selects one token and repeats this entire process to build complete sentences.

<p align="center">
  <img src="../lessons/images/select_probability.png" alt="Selecting Probabilities" width="400" height="200">
</p>

### What Temperature Does

**Temperature is a decimal value between 0 and 1** that directly influences these selection probabilities. It's like adjusting the "creativity dial" on Claude's responses.

<p align="center">
  <img src="../lessons/images/what_temp_does.png" alt="What Temperature Does" width="400" height="200">
</p>

**At low temperatures (near 0), Claude becomes very deterministic** - it almost always picks the highest probability token. **At high temperatures (near 1)**, Claude distributes probability more evenly across options, **leading to more varied and creative outputs**.

### Choosing the Right Temperature

Different tasks call for different temperature ranges:

<p align="center">
  <img src="../lessons/images/choosing_temp.png" alt="Choosing Temperature" width="400" height="300">
</p>

**Low Temperature (0.0 - 0.3)**
* Factual responses
* Coding assistance
* Data extraction
* Content moderation

**Medium Temperature (0.4 - 0.7)**
* Summarization
* Educational content
* Problem-solving
* Creative writing with constraints

**High Temperature (0.8 - 1.0)**
* Brainstorming
* Creative writing
* Marketing content
* Joke generation

### Implementing Temperature in Code

Adding temperature support to your chat function is straightforward. Here's how to modify your existing `chat` function:

In [23]:
def chat2(messages, system=None, temperature=1.0):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        # this will work with older (<1.1.0) SDK
        # "temperature": temperature,
        # for 1.1.0+ SDK use the following
        "extra_body": {"temperature": temperature},
    }

    if system is not None:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

The key changes are adding `temperature=1.0` as a parameter and including `"temperature": temperature` in the params dictionary (or `"extra_body": {"temperature": temperature}` for Anthropic SDK version >=1.1.0)

### Testing Temperature Effects

To see temperature in action, try generating movie ideas with different settings:

In [25]:
# Low temperature - more predictable
messages = [{"role": "user", "content": "Give me a one-sentence movie idea."}]

for i in range(3):
    # make repeated calls - with temperature=0.0, expect to
    # see similar responses (may/may-not be identical!). For
    # temperature=1.0, expect to see more creative responses on each call.

    answer = chat2(messages, temperature=0.0)
    console.print(f"[green]Assistant (low temp):[/green] {answer}")

    # High temperature - more creative
    answer = chat2(messages, temperature=1.0)
    console.print(f"[red]Assistant (high temp):[/red] {answer}")

    console.print("[yellow]\n=====================\n[/yellow]")

Assistant (low temp): A washed-up stunt double discovers that the action movie 
star he's doubled for is actually a spy, and he's mistaken for the agent during
a high-stakes international operation.
Assistant (high temp): A washed-up stunt performer discovers a secret camera in
their apartment and must figure out who's watching them before the mysterious 
observer decides to make them the star of their next "project."


Assistant (low temp): A washed-up stunt double discovers that the action movie 
star he's doubled for is actually a spy, and he's mistaken for the agent during
a high-stakes international operation.
Assistant (high temp): A skilled con artist discovers that the elaborate heist 
she's planning is actually being orchestrated by someone who's conning her, 
forcing her to decide whether to expose them or go deeper into the scheme for a
bigger score.


Assistant (low temp): A washed-up stunt double discovers that the action movie 
star he's doubled for is actually a spy, and

At temperature `0.0`, you _might_ consistently get responses like "A time-traveling archaeologist must prevent ancient artifacts from being stolen." At temperature `1.0`, you'll see much more variety in themes, characters, and plot elements.

### Key Takeaways

Remember that temperature doesn't guarantee different outputs - it just changes the probability of getting them. Even at high temperatures, Claude might occasionally produce similar responses. The key is matching your temperature choice to your specific use case:

* Need consistent, factual responses? Use low temperature
* Want creative brainstorming? Dial up the temperature
* Somewhere in between? Medium temperatures work well for most general tasks

Temperature is one of the most practical parameters you can adjust to fine-tune Claude's behavior for your specific needs.


## Response Streaming

When building chat applications with Claude, there's a significant user experience challenge: **responses can take 10-30 seconds to generate, leaving users staring at a loading spinner**. The solution is `response streaming`, which lets users see text appear chunk by chunk as Claude generates it, creating a much more responsive feel.

<p align="center">
  <img src="../lessons/images/stream1.png" alt="Streaming Need" width="450" height="300">
</p>

### The Problem with Standard Responses

In a typical chat setup, your server sends a user message to Claude and _waits for the complete response_ before sending anything back to the client. This creates an awkward delay where users have no feedback that anything is happening.

<p align="center">
  <img src="../lessons/images/stream2.png" alt="Streaming Problem" width="450" height="300">
</p>

### How Streaming Works

With streaming enabled, Claude _immediately sends back an initial response_ indicating it has received your request and is starting to generate text. Then you receive a series of events, each containing a small piece of the overall response.

<p align="center">
  <img src="../lessons/images/stream3.png" alt="With Streaming" width="450" height="300">
</p>

Your server can forward these text chunks to your client application as they arrive, allowing users to see the response building up word by word. All of these events are part of a single request to Claude.

<p align="center">
  <img src="../lessons/images/stream4.png" alt="With Streaming" width="450" height="300">
</p>

### Understanding Stream Events

When you enable streaming, Claude sends back several types of events:

* `MessageStart` - A new message is being sent
* `ContentBlockStart` - Start of a new block containing text, tool use, or other content
* `ContentBlockDelta` - Chunks of the actual generated text
* `ContentBlockStop` - The current content block has been completed
* `MessageDelta` - The current message is complete
* `MessageStop` - End of information about the current message

<p align="center">
  <img src="../lessons/images/stream_events.png" alt="Streaming Events" width="450" height="300">
</p>

The `ContentBlockDelta` events contain the actual generated text that you'll want to display to users.

### Basic Streaming Implementation

To enable streaming, add stream=True to your messages.create call:

In [26]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CfGRc5jhhdRb68FcMM5A5', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeDB', type='text_delta'), index=0, type='content_block_de

### Simplified Text Streaming

Rather than manually parsing events, you can use the SDK's simplified streaming interface that extracts just the text content, like so:

In [28]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=MODEL, max_tokens=MAX_TOKENS, messages=messages
) as stream:
    for text in stream.text_stream:
        console.print(text, end="")

# FakeDB

FakeDB is an in-memory database engine that generates randomized data structures and returns mock query results to simulate a fully functional relational database without requiring actual data persistence or schema validation.

This approach automatically filters out everything except the actual text content, which is usually what you need for displaying responses to users.

### Getting the Complete Message

While streaming individual chunks is great for user experience, you often need the complete message for storage or further processing. After streaming completes, you can get the assembled final message:

In [30]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=MODEL, max_tokens=MAX_TOKENS, messages=messages
) as stream:
    for text in stream.text_stream:
        # display the chunk as above
        print(text, end="")

    # Get the complete message for database storage
    final_message = stream.get_final_message()
    add_assistant_message(messages, final_message)

print(f"\n========\nFinal message:\n {final_message.content[0].text}")

console.print(f"[yellow]\n---\n[/yellow]Final messages list:\n{messages}")

# FakeDB

FakeDB is a lightweight in-memory database that generates randomized structured data for testing and development purposes, eliminating the need for real datasets in early-stage application development.
Final message:
 # FakeDB

FakeDB is a lightweight in-memory database that generates randomized structured data for testing and development purposes, eliminating the need for real datasets in early-stage application development.

---
Final messages list:
[{'role': 'user', 'content': 'Write a 1 sentence description of a fake 
database'}, {'role': 'assistant', 'content': 
ParsedMessage(id='msg_011CfGRtbypqmmABysD3VDKa', container=None, 
content=[ParsedTextBlock(citations=None, text='# FakeDB\n\nFakeDB is a 
lightweight in-memory database that generates randomized structured data for 
testing and development purposes, eliminating the need for real datasets in 
early-stage application development.', type='text', parsed_output=None)], 
model='claude-haiku-4-5-20251001', role='assista

This gives you the best of both worlds: real-time streaming for users and a complete message object for your application logic.

## Structured Data

When you need Claude to generate structured data like JSON, Python code, or bulleted lists, you'll often run into a common problem: Claude wants to be helpful and add explanatory text around your content. While this is usually great, sometimes you need just the raw data with nothing else.

Consider building a web app that generates AWS EventBridge rules. Users enter a description, click generate, and expect to see clean JSON they can immediately copy and use. If Claude returns the JSON wrapped in markdown code blocks with explanatory text, users can't simply copy the entire response - they have to manually select just the JSON portion.

<p align="center">
  <img src="../lessons/images/structured_data1.png" alt="Structured Data" width="275" height="350">
</p>

### The Problem with Default Responses

By default, when you ask Claude to generate JSON, you might get something like this (everything between the  `----` lines)

`----------`

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}
```

This rule captures EC2 instance state changes when instances start running.

`----------`

The JSON is correct, but it's wrapped in markdown formatting (```json ... ```) and includes explanatory text `This rule...`. For a web app where users need to copy the raw JSON, this creates friction in the user experience.

### The Solution?: Assistant Message Prefilling + Stop Sequences

You can combine assistant message prefilling with stop sequences to get exactly the content you want. Here's how it works:

```python
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
```

This technique works because:

* The user message tells Claude what to generate
* The prefilled assistant message makes Claude think it already started a markdown code block
* Claude continues by writing just the JSON content
* When Claude tries to close the code block with ```, the stop sequence immediately ends generation

<p align="center">
  <img src="../lessons/images/structured_data2.png" alt="Structured Data" height="450">
</p>

The result is clean JSON with no extra formatting:

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}
```

### Processing the Response

You might notice some extra newline characters in the response. These are easy to handle:

```python
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())
```

### Beyond JSON

This technique isn't limited to JSON generation. Use it anytime you need structured data without commentary:

* Python code snippets
* Bulleted lists
* CSV data
* Any formatted content where you want just the content, not explanations

The key is identifying what Claude naturally wants to wrap your content in, then using that as your prefill and stop sequence. For code, it's usually markdown code blocks. For lists, it might be different formatting markers.

This approach gives you precise control over Claude's output format, making it much easier to integrate AI-generated content into applications where clean, structured data is essential.

### Structured Data Exercise

* Use messaging prefilling and stop sequences _only_ to get three different commands in a single response
* There shouldn't be any comments or explanations
* **Hint:** message profiling isn't just limited to characters like ```

In [31]:
def chat3(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        # this will work with older (<1.1.0) SDK
        # "temperature": temperature,
        # for 1.1.0+ SDK use the following
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences,
    }

    if system is not None:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [32]:
messages = []

prompt = """
Generate 3 different sample AWS CLI commands. Each should be very short
"""

add_user_message(messages, prompt)
response = chat3(messages)
add_assistant_message(messages, response)
console.print(f"[green]Assistant:[/green] {response}")

Assistant: # 3 Sample AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **Get current AWS account ID:**
```bash
aws sts get-caller-identity
```


In [33]:
messages = []

prompt = """
Generate 3 different sample AWS CLI commands. Each should be very short
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")
response = chat3(messages, stop_sequences=["```"])
response.strip()
print(response)
# add_assistant_message(messages, response)
# console.print(f"[green]Assistant:[/green] {response}")


# 1. List all S3 buckets
aws s3 ls

# 2. Describe EC2 instances
aws ec2 describe-instances

# 3. Get Lambda function info
aws lambda get-function --function-name my-function

